In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import warnings

In [2]:
project_root = Path.cwd().parent

In [3]:
data_dir=project_root/"data"
raw_dir=data_dir/"raw"
processed_dir=data_dir/"processed"
external_dir=data_dir/"external"
interim_dir=data_dir/"interim"

In [4]:
print(raw_dir)

c:\Users\ujjwa\OneDrive\Documents\beauty-recommender\data\raw


In [5]:
raw_dir.glob("*.csv")

<generator object Path.glob at 0x000001EE98457840>

In [6]:
raw_dir.glob("reviews*.csv")

<generator object Path.glob at 0x000001EE98457990>

In [7]:
required_files={
    "product_info": raw_dir/"product_info.csv",
    "ingredient_master":raw_dir/"ingredient_master.csv"
}

In [8]:
review_files = sorted(raw_dir.glob("reviews*.csv"))

In [9]:
missing_files = []

for dataset_name, file_path in required_files.items():
    if not file_path.exists():
        missing_files.append((dataset_name, file_path))

In [10]:
if len(review_files) == 0:
    raise FileNotFoundError(
        "No review CSV files were found in the raw data directory."
    )

In [11]:
if missing_files:
    error_message = "\n".join(
        f"- {dataset}: {path}"
        for dataset, path in missing_files
    )

    raise FileNotFoundError(
        f"Missing required datasets:\n{error_message}"
    )

In [12]:
review_files

[WindowsPath('c:/Users/ujjwa/OneDrive/Documents/beauty-recommender/data/raw/reviews_0-250.csv'),
 WindowsPath('c:/Users/ujjwa/OneDrive/Documents/beauty-recommender/data/raw/reviews_1250-end.csv'),
 WindowsPath('c:/Users/ujjwa/OneDrive/Documents/beauty-recommender/data/raw/reviews_250-500.csv'),
 WindowsPath('c:/Users/ujjwa/OneDrive/Documents/beauty-recommender/data/raw/reviews_500-750.csv'),
 WindowsPath('c:/Users/ujjwa/OneDrive/Documents/beauty-recommender/data/raw/reviews_750-1250.csv')]

In [13]:
def load_csv(file_path:Path) -> pd.DataFrame:
    try:
        df=pd.read_csv(file_path)
        return df
    except Exception as e:
        raise RuntimeError(
            f"failed to load csv file: {file_path}"
        ) from e
    



product_df=load_csv(required_files["product_info"])

ingredients_df=load_csv(required_files["ingredient_master"])


reviews_df=[
    load_csv(file_path)
    for file_path in review_files
]

C:\Users\ujjwa\AppData\Local\Temp\ipykernel_18888\2471274204.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(file_path)
C:\Users\ujjwa\AppData\Local\Temp\ipykernel_18888\2471274204.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(file_path)
C:\Users\ujjwa\AppData\Local\Temp\ipykernel_18888\2471274204.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(file_path)


In [14]:
product_df.shape


(8494, 27)

In [15]:
ingredients_df.shape

(30, 10)

In [17]:
for i ,df in enumerate(reviews_df,start=1):
    print(f"Review File{i}: {df.shape}")

Review File1: (602130, 19)
Review File2: (49977, 19)
Review File3: (206725, 19)
Review File4: (116262, 19)
Review File5: (119317, 19)


In [18]:
# Column names of the first review dataset
reference_columns = reviews_df[0].columns

# Compare every other review file
for i, df in enumerate(reviews_df[1:], start=2):
    if reference_columns.equals(df.columns):
        print(f"✅ Review File {i}: Schema matches")
    else:
        print(f"❌ Review File {i}: Schema mismatch")

✅ Review File 2: Schema matches
✅ Review File 3: Schema matches
✅ Review File 4: Schema matches
✅ Review File 5: Schema matches


In [19]:
review_dfs=pd.concat(
    reviews_df,
    ignore_index=True
)

In [20]:
review_dfs.shape

(1094411, 19)

In [110]:
review_dfs.head()

,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0
1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
3,6083038851,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,Can't go wrong with any of them,NaN,brown,combination,black,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
4,47056667835,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",A must have !!!,light,hazel,combination,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0


In [21]:
review_dfs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1094411 entries, 0 to 1094410
Data columns (total 19 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   Unnamed: 0                1094411 non-null  int64  
 1   author_id                 1094411 non-null  object 
 2   rating                    1094411 non-null  int64  
 3   is_recommended            926423 non-null   float64
 4   helpfulness               532819 non-null   float64
 5   total_feedback_count      1094411 non-null  int64  
 6   total_neg_feedback_count  1094411 non-null  int64  
 7   total_pos_feedback_count  1094411 non-null  int64  
 8   submission_time           1094411 non-null  object 
 9   review_text               1092967 non-null  object 
 10  review_title              783757 non-null   object 
 11  skin_tone                 923872 non-null   object 
 12  eye_color                 884783 non-null   object 
 13  skin_type                 9

In [22]:
review_dfs.head()

,Unnamed: 0,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
0,0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0
1,1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
2,2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
3,3,6083038851,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,Can't go wrong with any of them,NaN,brown,combination,black,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
4,4,47056667835,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",A must have !!!,light,hazel,combination,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0


In [23]:
review_dfs.sample(5, random_state=42)

,Unnamed: 0,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
1049894,74800,5839472814,5,1.0,1.0,4,0,4,2022-08-14,"I have really sensitive skin, and this is the ...",NaN,fairLight,brown,combination,brown,P405944,Ultra Sun Protection Lotion Broad Spectrum SPF...,Shiseido,42.0
619554,17424,30632543234,5,1.0,0.0,17,17,0,2021-03-06,"Lightweight, smells amazing! It worked great u...",So good!,light,hazel,normal,brown,P455618,Full Spectrum 360° Refreshing Water Mist Organ...,COOLA,36.0
634782,32652,1212678959,5,1.0,0.8,10,2,8,2020-04-27,I don’t know how anyone could not like this ma...,Leaves You Glowing,light,hazel,combination,brown,P433522,Goddess Clay Mask,Charlotte Tilbury,58.0
706815,54708,32052969220,5,1.0,NaN,0,0,0,2021-06-01,I’m in love with this product. My skin looks g...,SK-II,fair,blue,combination,black,P375853,Facial Treatment Clear Lotion Toner,SK-II,80.0
731656,79549,1135020217,5,1.0,NaN,0,0,0,2018-09-29,I got samples of this cleasner from Sunday Ril...,Great cleasner with a better price!,light,blue,combination,brown,P309310,Ceramic Slip French Green Clay Cleanser,Sunday Riley,35.0


In [24]:
review_dfs.columns.to_list()

['Unnamed: 0',
 'author_id',
 'rating',
 'is_recommended',
 'helpfulness',
 'total_feedback_count',
 'total_neg_feedback_count',
 'total_pos_feedback_count',
 'submission_time',
 'review_text',
 'review_title',
 'skin_tone',
 'eye_color',
 'skin_type',
 'hair_color',
 'product_id',
 'product_name',
 'brand_name',
 'price_usd']

In [25]:
expected_rows = sum(df.shape[0] for df in reviews_df)

print(f"Expected Rows : {expected_rows}")
print(f"Actual Rows   : {review_dfs.shape[0]}")
print(f"Rows Match    : {expected_rows == review_dfs.shape[0]}")

Expected Rows : 1094411
Actual Rows   : 1094411
Rows Match    : True


schema validation

In [29]:
product_df.columns

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'variation_desc', 'ingredients', 'price_usd', 'value_price_usd',
       'sale_price_usd', 'limited_edition', 'new', 'online_only',
       'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price'],
      dtype='object')

In [26]:
EXPECTED_PRODUCT_COLUMNS = {
    "product_id",
    "product_name",
    "brand_id",
    "brand_name",
    "loves_count",
    "rating",
    "reviews",
    "size",
    "variation_type",
    "variation_value",
    "variation_desc",
    "ingredients",
    "price_usd",
    "value_price_usd",
    "sale_price_usd",
    "limited_edition",
    "new",
    "online_only",
    "out_of_stock",
    "sephora_exclusive",
    "highlights",
    "primary_category",
    "secondary_category",
    "tertiary_category",
    "child_count",
    "child_max_price",
    "child_min_price",
}

In [28]:
ingredients_df.columns

Index(['ingredient', 'category', 'primary_function', 'secondary_function',
       'suitable_skin', 'suitable_concerns', 'avoid_with', 'pregnancy_safe',
       'vegan', 'irritation_score'],
      dtype='object')

In [30]:
EXPECTED_INGREDIENT_COLUMNS = {
    "ingredient",
    "category",
    "primary_function",
    "secondary_function",
    "suitable_skin",
    "suitable_concerns",
    "avoid_with",
    "vegan",
    "irritation_score",
    "pregnancy_safe"
}

In [32]:
print(review_dfs.columns)

Index(['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness',
       'total_feedback_count', 'total_neg_feedback_count',
       'total_pos_feedback_count', 'submission_time', 'review_text',
       'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color',
       'product_id', 'product_name', 'brand_name', 'price_usd'],
      dtype='object')


In [46]:
EXPECTED_REVIEW_COLUMNS = {
   "author_id", 
   "rating", 
   "is_recommended", 
   "helpfulness",
   "total_feedback_count", 
   "total_neg_feedback_count",
    "total_pos_feedback_count", 
    "submission_time", 
    "review_text",
    "review_title", 
    "skin_tone", 
    "eye_color", 
    "skin_type", 
    "hair_color",
    "product_id", 
    "product_name", 
    "brand_name", 
    "price_usd",
    "Unnamed: 0"
}

In [34]:
def validate_columns(
    df: pd.DataFrame,
    expected_columns: set,
    dataset_name: str
) -> None:
    """
    Validate that a dataset contains the expected columns.
    """

    actual_columns = set(df.columns)

    missing_columns = expected_columns - actual_columns
    unexpected_columns = actual_columns - expected_columns

    print(f"\n{'='*60}")
    print(f"Schema Validation: {dataset_name}")
    print(f"{'='*60}")

    if not missing_columns and not unexpected_columns:
        print("✅ Schema validation passed.")
        return

    if missing_columns:
        print("\n❌ Missing Columns:")
        for col in sorted(missing_columns):
            print(f"   - {col}")

    if unexpected_columns:
        print("\n⚠️ Unexpected Columns:")
        for col in sorted(unexpected_columns):
            print(f"   - {col}")

    raise ValueError(f"{dataset_name} schema validation failed.")

In [36]:
validate_columns(
    product_df,
    EXPECTED_PRODUCT_COLUMNS,
    "Products"
)


Schema Validation: Products
✅ Schema validation passed.


In [37]:
validate_columns(
    ingredients_df,
    EXPECTED_INGREDIENT_COLUMNS,
    "Ingredients"
)


Schema Validation: Ingredients
✅ Schema validation passed.


In [47]:
validate_columns(
    review_dfs,
    EXPECTED_REVIEW_COLUMNS,
    "Reviews"
)


Schema Validation: Reviews
✅ Schema validation passed.


In [44]:
type(review_dfs)

pandas.core.frame.DataFrame

In [48]:
print("=" * 60)
print("Products Data Types")
print("=" * 60)

print(product_df.dtypes)

Products Data Types
product_id             object
product_name           object
brand_id                int64
brand_name             object
loves_count             int64
rating                float64
reviews               float64
size                   object
variation_type         object
variation_value        object
variation_desc         object
ingredients            object
price_usd             float64
value_price_usd       float64
sale_price_usd        float64
limited_edition         int64
new                     int64
online_only             int64
out_of_stock            int64
sephora_exclusive       int64
highlights             object
primary_category       object
secondary_category     object
tertiary_category      object
child_count             int64
child_max_price       float64
child_min_price       float64
dtype: object


In [49]:

print(ingredients_df.dtypes)

ingredient            object
category              object
primary_function      object
secondary_function    object
suitable_skin         object
suitable_concerns     object
avoid_with            object
pregnancy_safe        object
vegan                 object
irritation_score       int64
dtype: object


In [51]:
print(review_dfs.dtypes)

Unnamed: 0                    int64
author_id                    object
rating                        int64
is_recommended              float64
helpfulness                 float64
total_feedback_count          int64
total_neg_feedback_count      int64
total_pos_feedback_count      int64
submission_time              object
review_text                  object
review_title                 object
skin_tone                    object
eye_color                    object
skin_type                    object
hair_color                   object
product_id                   object
product_name                 object
brand_name                   object
price_usd                   float64
dtype: object


In [57]:
def summarize_dtypes(
    df: pd.DataFrame,
    dataset_name: str
) -> pd.DataFrame:
    """
    Return a summary of column data types.
    """

    dtype_summary = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Non-Null Count": df.count().values,
        "Missing Values": df.isna().sum().values,
    })

    print(f"\n{'=' * 60}")
    print(f"{dataset_name} - Data Type Summary")
    print(f"{'=' * 60}")

    return dtype_summary

In [61]:
product_dtype_summary = summarize_dtypes(
    product_df,
    "Products"
)
product_dtype_summary



Products - Data Type Summary


,Column,Data Type,Non-Null Count,Missing Values
0,product_id,object,8494,0
1,product_name,object,8494,0
2,brand_id,int64,8494,0
3,brand_name,object,8494,0
4,loves_count,int64,8494,0
5,rating,float64,8216,278
6,reviews,float64,8216,278
7,size,object,6863,1631
8,variation_type,object,7050,1444
9,variation_value,object,6896,1598


In [62]:
ingredient_dtype_summary = summarize_dtypes(
    ingredients_df,
    "Ingredients"
)

ingredient_dtype_summary


Ingredients - Data Type Summary


,Column,Data Type,Non-Null Count,Missing Values
0,ingredient,object,30,0
1,category,object,30,0
2,primary_function,object,30,0
3,secondary_function,object,30,0
4,suitable_skin,object,30,0
5,suitable_concerns,object,30,0
6,avoid_with,object,11,19
7,pregnancy_safe,object,30,0
8,vegan,object,30,0
9,irritation_score,int64,30,0


In [63]:
review_dtype_summary = summarize_dtypes(
    review_dfs,
    "Reviews"
)
review_dtype_summary


Reviews - Data Type Summary


,Column,Data Type,Non-Null Count,Missing Values
0,Unnamed: 0,int64,1094411,0
1,author_id,object,1094411,0
2,rating,int64,1094411,0
3,is_recommended,float64,926423,167988
4,helpfulness,float64,532819,561592
5,total_feedback_count,int64,1094411,0
6,total_neg_feedback_count,int64,1094411,0
7,total_pos_feedback_count,int64,1094411,0
8,submission_time,object,1094411,0
9,review_text,object,1092967,1444


In [64]:
review_dfs["Unnamed: 0"].head()

0    0
1    1
2    2
3    3
4    4
Name: Unnamed: 0, dtype: int64

In [65]:
review_dfs["Unnamed: 0"].nunique()

602130

In [67]:
review_dfs["Unnamed: 0"].tail()

1094406    119312
1094407    119313
1094408    119314
1094409    119315
1094410    119316
Name: Unnamed: 0, dtype: int64

In [68]:
def remove_unnamed_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove automatically generated unnamed columns,
    typically created when saving CSV files with the index.
    """

    return df.loc[:, ~df.columns.str.startswith("Unnamed")]

In [69]:
review_dfs = remove_unnamed_columns(review_dfs)

In [70]:
review_dfs.columns

Index(['author_id', 'rating', 'is_recommended', 'helpfulness',
       'total_feedback_count', 'total_neg_feedback_count',
       'total_pos_feedback_count', 'submission_time', 'review_text',
       'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color',
       'product_id', 'product_name', 'brand_name', 'price_usd'],
      dtype='object')

Production Data Quality Report

In [71]:
def generate_missing_value_report(
        df:pd.DataFrame,
        dataset_name:str
) -> pd.DataFrame:
    
    """
    Generate a missing value report for a dataset.
    """

    report=pd.DataFrame({
        "Column":df.columns,
        "Missing Count":df.isna().sum().values 
    })

    report["Missing %"]=(
        report["Missing Count"]/len(df)*100
    ).round(2)

    def classify_severity(pct):
        if pct == 0:
            return "None"
        elif pct <= 5:
            return "Low"
        elif pct <= 20:
            return "Medium"
        else:
            return "High"

    report["Severity"] = report["Missing %"].apply(
        classify_severity
    )

    report = report.sort_values(
        by="Missing %",
        ascending=False
    ).reset_index(drop=True)

    print(f"\n{'='*70}")
    print(f"{dataset_name} - Missing Value Report")
    print(f"{'='*70}")

    return report



In [73]:
product_missing_report = generate_missing_value_report(
    product_df,
    "Products"
)
product_missing_report



Products - Missing Value Report


,Column,Missing Count,Missing %,Severity
0,sale_price_usd,8224,96.82,High
1,value_price_usd,8043,94.69,High
2,variation_desc,7244,85.28,High
3,child_max_price,5740,67.58,High
4,child_min_price,5740,67.58,High
5,highlights,2207,25.98,High
6,size,1631,19.20,Medium
7,variation_value,1598,18.81,Medium
8,variation_type,1444,17.00,Medium
9,tertiary_category,990,11.66,Medium


In [74]:
ingredient_missing_report = generate_missing_value_report(
    ingredients_df,
    "Ingredients"
)

ingredient_missing_report


Ingredients - Missing Value Report


,Column,Missing Count,Missing %,Severity
0,avoid_with,19,63.33,High
1,ingredient,0,0.00,None
2,category,0,0.00,None
3,primary_function,0,0.00,None
4,suitable_skin,0,0.00,None
5,secondary_function,0,0.00,None
6,suitable_concerns,0,0.00,None
7,pregnancy_safe,0,0.00,None
8,vegan,0,0.00,None
9,irritation_score,0,0.00,None


In [75]:
review_missing_report = generate_missing_value_report(
    review_dfs,
    "Reviews"
)
review_missing_report


Reviews - Missing Value Report


,Column,Missing Count,Missing %,Severity
0,helpfulness,561592,51.31,High
1,review_title,310654,28.39,High
2,hair_color,226768,20.72,High
3,eye_color,209628,19.15,Medium
4,skin_tone,170539,15.58,Medium
5,is_recommended,167988,15.35,Medium
6,skin_type,111557,10.19,Medium
7,review_text,1444,0.13,Low
8,rating,0,0.00,None
9,author_id,0,0.00,None


Duplicate Analysis

Exact Duplicate Function

In [76]:
def check_exact_duplicates(
        df:pd.DataFrame,
        dataset_name:str
):
    
    """
    Check for completely duplicated rows.
    """

    duplicate_count=df.duplicated().sum()

    duplicate_pct=(
        duplicate_count/len(df)
    )* 100
    print("=" * 60)
    print(dataset_name)
    print("=" * 60)

    print(f"Rows               : {len(df):,}")
    print(f"Exact Duplicates   : {duplicate_count:,}")
    print(f"Duplicate %        : {duplicate_pct:.2f}%")  
    

In [78]:
check_exact_duplicates(
    product_df,
    "Products"
)

Products
Rows               : 8,494
Exact Duplicates   : 0
Duplicate %        : 0.00%


In [79]:
check_exact_duplicates(
    ingredients_df,
    "Ingredients"
)

Ingredients
Rows               : 30
Exact Duplicates   : 0
Duplicate %        : 0.00%


In [80]:
check_exact_duplicates(
    review_dfs,
    "Reviews"
)

Reviews
Rows               : 1,094,411
Exact Duplicates   : 224
Duplicate %        : 0.02%


In [81]:
review_duplicates = review_dfs[
    review_dfs.duplicated(keep=False)
].sort_values(
    by=["author_id", "product_id"]
)

review_duplicates

,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
994984,909571449,5,1.0,1.0,1,0,1,2021-01-06,This moisturizer is hydrating and also gives m...,Hydrating and brightening,light,blue,dry,brown,P467750,Resveratrol Lift Retinol Alternative Firming C...,Caudalie,69.0
994985,909571449,5,1.0,1.0,1,0,1,2021-01-06,This moisturizer is hydrating and also gives m...,Hydrating and brightening,light,blue,dry,brown,P467750,Resveratrol Lift Retinol Alternative Firming C...,Caudalie,69.0
8173,926853990,5,1.0,NaN,0,0,0,2019-11-22,Holy cow! Couldn’t love this product more. I...,Worth its weight in gold!!,mediumTan,brown,combination,brown,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
8174,926853990,5,1.0,NaN,0,0,0,2019-11-22,Holy cow! Couldn’t love this product more. I...,Worth its weight in gold!!,mediumTan,brown,combination,brown,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
31808,995832198,5,NaN,NaN,0,0,0,2010-07-30,One of if not the best softener and healer I h...,Top Moisturizer,light,NaN,dry,NaN,P218700,100 percent Pure Argan Oil,Josie Maran,49.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
661635,9531043097,5,1.0,1.0,1,0,1,2020-02-16,I can’t get enough of Drunk Elephant products....,Another Drunk Elephant that doesn’t disappoint...,medium,brown,oily,black,P454120,F-Balm Electrolyte Waterfacial Mask,Drunk Elephant,54.0
661636,9531043097,5,1.0,1.0,1,0,1,2020-02-16,I can’t get enough of Drunk Elephant products....,Another Drunk Elephant that doesn’t disappoint...,medium,brown,oily,black,P454120,F-Balm Electrolyte Waterfacial Mask,Drunk Elephant,54.0
335278,969898278,5,NaN,NaN,0,0,0,2013-12-11,This stuff has changed my life..and my skin! I...,liquid gold,NaN,NaN,NaN,NaN,P376726,Maracuja Oil,tarte,48.0
335279,969898278,5,NaN,NaN,0,0,0,2013-12-11,This stuff has changed my life..and my skin! I...,liquid gold,NaN,NaN,NaN,NaN,P376726,Maracuja Oil,tarte,48.0


In [82]:
print("Duplicate rows:", len(review_duplicates))

Duplicate rows: 373


Review Dataset

Exact duplicate rows detected: 224 (0.02%)

Inspection indicates these are true duplicate records rather than distinct user interactions.

These duplicates will be removed before downstream feature engineering and modeling to prevent bias in engagement metrics and recommendation models.

In [83]:
review_duplicates.groupby(
    list(review_dfs.columns)
).size().value_counts()

2    16
Name: count, dtype: int64

In [84]:
review_duplicates.groupby(
    list(review_dfs.columns),
    dropna=False
).size().value_counts()

2     127
3      11
6       3
4       3
5       1
25      1
11      1
8       1
7       1
Name: count, dtype: int64

In [85]:
duplicate_summary = (
    review_duplicates
    .groupby(
        list(review_dfs.columns),
        dropna=False
    )
    .size()
)

duplicate_summary.value_counts().sort_index()

2     127
3      11
4       3
5       1
6       3
7       1
8       1
11      1
25      1
Name: count, dtype: int64

In [86]:
duplicate_summary.sum()

np.int64(373)

In [87]:
(duplicate_summary - 1).sum()

np.int64(224)

Business Key Validation

In [89]:
product_duplicate_ids = product_df[
    product_df["product_id"].duplicated(keep=False)
].sort_values("product_id")

print("=" * 60)
print("Product ID Validation")
print("=" * 60)
print(f"Duplicate Product IDs: {len(product_duplicate_ids)}")

Product ID Validation
Duplicate Product IDs: 0


In [90]:
ingredient_duplicate_names = ingredients_df[
    ingredients_df["ingredient"].duplicated(keep=False)
].sort_values("ingredient")

print("=" * 60)
print("Ingredient Validation")
print("=" * 60)
print(f"Duplicate Ingredients: {len(ingredient_duplicate_names)}")

Ingredient Validation
Duplicate Ingredients: 0


In [91]:
user_product_duplicates = review_dfs[
    review_dfs.duplicated(
        subset=["author_id", "product_id"],
        keep=False
    )
].sort_values(
    ["author_id", "product_id", "submission_time"]
)

print("=" * 60)
print("User-Product Review Validation")
print("=" * 60)
print(f"Duplicate User-Product Pairs: {len(user_product_duplicates)}")

User-Product Review Validation
Duplicate User-Product Pairs: 10471


In [92]:
user_product_duplicates[
    ["author_id",
     "product_id",
     "submission_time",
     "rating",
     "review_title"]
].head(20)

,author_id,product_id,submission_time,rating,review_title
28172,1743286,P218700,2013-09-10,5,Great product
27496,1743286,P218700,2014-05-14,5,Love this stuff
31360,6527230,P218700,2010-12-26,5,Amazing...
30937,6527230,P218700,2011-04-01,5,Second bottle and loving it!!
23903,13038147,P7880,2011-03-06,5,Great for sensitive skin!
23554,13038147,P7880,2011-11-17,5,Takes off eye make up!
22563,13929258,P7880,2013-11-22,5,My third tube!
22322,13929258,P7880,2014-04-06,5,My fourth tube.
23915,23576611,P7880,2011-02-27,5,Best cleanser ever
23383,23576611,P7880,2012-02-13,5,The best cleanser ever


In [93]:
sample_author = user_product_duplicates.iloc[0]["author_id"]
sample_product = user_product_duplicates.iloc[0]["product_id"]

user_product_duplicates[
    (user_product_duplicates["author_id"] == sample_author) &
    (user_product_duplicates["product_id"] == sample_product)
].sort_values("submission_time")

,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
28172,1743286,5,NaN,1.0,1,0,1,2013-09-10,I’ve been using this oil for several years. I ...,Great product,NaN,NaN,combination,NaN,P218700,100 percent Pure Argan Oil,Josie Maran,49.0
27496,1743286,5,NaN,NaN,0,0,0,2014-05-14,I’ve been using this oil for several years. I ...,Love this stuff,NaN,NaN,NaN,NaN,P218700,100 percent Pure Argan Oil,Josie Maran,49.0


foreign key validation

In [94]:
#convert to sets
product_ids = set(product_df["product_id"])

review_product_ids = set(review_dfs["product_id"])

In [95]:
print("=" * 60)
print("Foreign Key Validation")
print("=" * 60)

print(f"Products in catalog        : {len(product_ids):,}")
print(f"Products in reviews        : {len(review_product_ids):,}")

Foreign Key Validation
Products in catalog        : 8,494
Products in reviews        : 2,351


Find Orphan Reviews:
Which products appear in reviews but do not exist in the product catalog?



In [96]:
orphan_product_ids=review_product_ids-product_ids

print(f"orphan product ids: {len(orphan_product_ids):,}")

orphan product ids: 0


Products without reviews

In [97]:
products_without_reviews = product_ids - review_product_ids

print(f"Products without reviews   : {len(products_without_reviews):,}")

Products without reviews   : 6,143


In [98]:
catalog_coverage = (
    len(review_product_ids & product_ids)
    / len(product_ids)
) * 100

print(f"Catalog Review Coverage    : {catalog_coverage:.2f}%")

Catalog Review Coverage    : 27.68%


domain validation:


Numeric Range Validation

In [99]:
def validate_numeric_range(
        df,
        column,
        min_value=None,
        max_value=None
):
     """
    Validate whether numeric values fall within an expected range.
    """
     invalid_mask=pd.Series(False,index=df.index)  #Currently every row is assumed valid

     if min_value is not None:
          invalid_mask |=df[column]< min_value
     if max_value is not None:
          invalid_mask |= df[column]> max_value
     invalid_count=invalid_mask.sum()

     print(f"{column:<30} Invalid Values: {invalid_count:,}")

     return invalid_mask


In [100]:
validate_numeric_range(
    product_df,
    column="price_usd",
    min_value=0
)



price_usd                      Invalid Values: 0


0       False
1       False
2       False
3       False
4       False
        ...  
8489    False
8490    False
8491    False
8492    False
8493    False
Length: 8494, dtype: bool

In [101]:
validate_numeric_range(
    product_df,
    column="rating",
    min_value=0,
    max_value=5
)



rating                         Invalid Values: 0


0       False
1       False
2       False
3       False
4       False
        ...  
8489    False
8490    False
8491    False
8492    False
8493    False
Length: 8494, dtype: bool

In [102]:
validate_numeric_range(
    product_df,
    column="reviews",
    min_value=0
)

reviews                        Invalid Values: 0


0       False
1       False
2       False
3       False
4       False
        ...  
8489    False
8490    False
8491    False
8492    False
8493    False
Length: 8494, dtype: bool

Binary Flag Validation

In [103]:
def validate_binary_column(df, column):
    """
    Validate that a binary column contains only 0 or 1.
    """

    valid_values = {0, 1}

    invalid_mask = ~df[column].isin(valid_values)

    invalid_count = invalid_mask.sum()

    print(f"{column:<30} Invalid Values: {invalid_count:,}")

    return invalid_mask

In [105]:
binary_columns = [
    "limited_edition",
    "new",
    "online_only",
    "out_of_stock",
    "sephora_exclusive"
]

for column in binary_columns:
    validate_binary_column(product_df, column)

limited_edition                Invalid Values: 0
new                            Invalid Values: 0
online_only                    Invalid Values: 0
out_of_stock                   Invalid Values: 0
sephora_exclusive              Invalid Values: 0


memory optimization

In [106]:
product_df["product_id"] = product_df["product_id"].astype("string")
product_df["brand_name"] = product_df["brand_name"].astype("category")
product_df["primary_category"] = product_df["primary_category"].astype("category")

product_df["price_usd"] = pd.to_numeric(product_df["price_usd"], errors="coerce")
product_df["rating"] = pd.to_numeric(product_df["rating"], errors="coerce")

In [107]:
review_dfs["author_id"] = review_dfs["author_id"].astype("string")
review_dfs["product_id"] = review_dfs["product_id"].astype("string")

review_dfs["rating"] = review_dfs["rating"].astype("int8")
review_dfs["total_feedback_count"] = review_dfs["total_feedback_count"].astype("int32")

In [108]:
ingredients_df["ingredient"] = ingredients_df["ingredient"].astype("category")
ingredients_df["category"] = ingredients_df["category"].astype("category")

In [111]:
product_df.to_csv(processed_dir / "products_clean.csv", index=False)
review_dfs.to_csv(processed_dir / "reviews_clean.csv", index=False)
ingredients_df.to_csv(processed_dir / "ingredients_clean.csv", index=False)